# SNR-Auswertung fuer gankan-lwf

Notebook-Version analog zur bestehenden SNR-Analyse aus metricgan-lwf.

- Erwartete Run-Struktur: results/results/hparams/<run_name>[/<seed>]/enhanced_wavs
- Fokus auf deine 4G-Runs: train_mgk_g4_d4*
- Baseline: train_mgk_g4_d4
- Ausgabe: Summary- und Detail-CSV + Plots

In [ ]:
from __future__ import annotations

import json
import math
import re
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import pandas as pd
import torch
import torchaudio

plt.style.use('seaborn-v0_8-whitegrid')
EPS = 1e-10

In [ ]:
# Konfiguration
from pathlib import Path

RESULTS_ROOT = Path('results/results/hparams')
BASELINE_RUN = 'train_mgk_g4_d4'
RUN_PATTERN = 'train_mgk_g4_d4*'

# Optional manuell setzen, falls aus hyperparams nicht aufloesbar
ANNOTATION_OVERRIDE = None  # z.B. Path('/data/noisy-vctk-16k/test.json')
DATA_ROOT_OVERRIDE = None   # z.B. Path('/data/noisy-vctk-16k')

SUMMARY_CSV = RESULTS_ROOT / 'snr_summary.csv'
DETAILS_CSV = RESULTS_ROOT / 'snr_details.csv'

In [ ]:
@dataclass
class ItemPaths:
    utt_id: str
    noisy: Path
    clean: Path


def _extract_hparam(text: str, key: str) -> Optional[str]:
    pattern = re.compile(rf'^\s*{re.escape(key)}\s*:\s*(.+?)\s*$', re.MULTILINE)
    m = pattern.search(text)
    if not m:
        return None
    return m.group(1).strip().strip("\"'")


def parse_paths_from_hyperparams(hparams_file: Path) -> Tuple[Optional[str], Optional[str]]:
    if not hparams_file.is_file():
        return None, None
    text = hparams_file.read_text(encoding='utf-8', errors='ignore')
    data_folder = _extract_hparam(text, 'data_folder')
    test_ann = _extract_hparam(text, 'test_annotation')
    return data_folder, test_ann


def resolve_template_path(path_str: Optional[str], data_root: Optional[str]) -> Optional[Path]:
    if path_str is None:
        return None

    value = path_str
    if data_root:
        value = value.replace('{data_root}', data_root)

    ref_match = re.search(r'<data_folder>(.+)$', value)
    if ref_match and data_root:
        suffix = ref_match.group(1).lstrip('/')
        value = str(Path(data_root) / suffix)

    cleaned = value.replace('!ref', '').strip().strip("\"'")
    return Path(cleaned).expanduser()


def load_annotation_pairs(annotation_file: Path, data_root: Optional[str]) -> List[ItemPaths]:
    with annotation_file.open('r', encoding='utf-8') as f:
        payload = json.load(f)

    items: List[ItemPaths] = []
    for utt_id, row in payload.items():
        noisy = str(row.get('noisy_wav', '')).strip()
        clean = str(row.get('clean_wav', '')).strip()
        if not noisy or not clean:
            continue

        if data_root:
            noisy = noisy.replace('{data_root}', data_root)
            clean = clean.replace('{data_root}', data_root)

        items.append(ItemPaths(utt_id=utt_id, noisy=Path(noisy), clean=Path(clean)))
    return items


def read_mono(path: Path) -> Tuple[torch.Tensor, int]:
    wav, sr = torchaudio.load(str(path))
    if wav.numel() == 0:
        raise ValueError(f'Leere WAV-Datei: {path}')
    if wav.size(0) > 1:
        wav = wav.mean(dim=0, keepdim=True)
    return wav.squeeze(0).to(torch.float64), int(sr)


def snr_db(clean: torch.Tensor, degraded: torch.Tensor, eps: float = EPS) -> float:
    length = min(clean.numel(), degraded.numel())
    if length == 0:
        return float('nan')
    c = clean[:length]
    d = degraded[:length]
    n = d - c
    p_sig = float(torch.mean(c * c).item())
    p_noise = float(torch.mean(n * n).item())
    return 10.0 * math.log10((p_sig + eps) / (p_noise + eps))


def _parse_epoch_from_stem(stem: str) -> int:
    for pat in (r'@(?P<e>\d+)$', r'epoch[_-]?(?P<e>\d+)$', r'_(?P<e>\d+)$'):
        m = re.search(pat, stem)
        if m:
            return int(m.group('e'))
    return -1


def index_enhanced_wavs(enhanced_dir: Path) -> Tuple[Dict[str, List[Path]], Dict[str, List[Path]]]:
    by_exact: Dict[str, List[Path]] = {}
    by_base: Dict[str, List[Path]] = {}
    for wav_path in sorted(enhanced_dir.rglob('*.wav')):
        stem = wav_path.stem
        by_exact.setdefault(stem, []).append(wav_path)
        base = stem.split('@', 1)[0]
        by_base.setdefault(base, []).append(wav_path)
    return by_exact, by_base


def choose_best(paths: List[Path]) -> Optional[Path]:
    if not paths:
        return None
    return sorted(paths, key=lambda p: (_parse_epoch_from_stem(p.stem), p.name))[-1]


def resolve_enhanced_path(utt_id: str, by_exact: Dict[str, List[Path]], by_base: Dict[str, List[Path]]) -> Optional[Path]:
    stem = Path(str(utt_id).strip()).stem
    if stem in by_exact:
        return choose_best(by_exact[stem])
    if stem in by_base:
        return choose_best(by_base[stem])
    return None


def discover_enhanced_dir(run_dir: Path) -> Optional[Path]:
    candidates = [run_dir / 'enhanced_wavs']
    if run_dir.is_dir():
        for child in sorted(run_dir.iterdir()):
            if child.is_dir():
                candidates.append(child / 'enhanced_wavs')
    existing = [p for p in candidates if p.is_dir()]
    if not existing:
        return None
    return max(existing, key=lambda p: len(list(p.rglob('*.wav'))))


def summarize(values: List[float]) -> Dict[str, float]:
    if not values:
        return {
            'count': 0,
            'mean': float('nan'),
            'median': float('nan'),
            'std': float('nan'),
            'min': float('nan'),
            'max': float('nan'),
        }
    t = torch.tensor(values, dtype=torch.float64)
    return {
        'count': int(t.numel()),
        'mean': float(torch.mean(t).item()),
        'median': float(torch.median(t).item()),
        'std': float(torch.std(t, unbiased=False).item()),
        'min': float(torch.min(t).item()),
        'max': float(torch.max(t).item()),
    }


def evaluate_run(run_name: str, run_dir: Path, items: List[ItemPaths]):
    enh_dir = discover_enhanced_dir(run_dir)
    if enh_dir is None:
        raise FileNotFoundError(f'Kein enhanced_wavs gefunden in {run_dir}')

    by_exact, by_base = index_enhanced_wavs(enh_dir)

    rows = []
    snr_in_values = []
    snr_out_values = []
    delta_values = []
    missing = 0
    failed = 0

    for item in items:
        enh_path = resolve_enhanced_path(item.utt_id, by_exact, by_base)
        if enh_path is None:
            missing += 1
            continue

        try:
            clean, sr_c = read_mono(item.clean)
            noisy, sr_n = read_mono(item.noisy)
            enh, sr_e = read_mono(enh_path)

            if sr_n != sr_c:
                noisy = torchaudio.functional.resample(noisy, sr_n, sr_c)
            if sr_e != sr_c:
                enh = torchaudio.functional.resample(enh, sr_e, sr_c)

            snr_in = snr_db(clean, noisy)
            snr_out = snr_db(clean, enh)
            delta = snr_out - snr_in

            snr_in_values.append(snr_in)
            snr_out_values.append(snr_out)
            delta_values.append(delta)

            rows.append({
                'run': run_name,
                'utt_id': item.utt_id,
                'clean_path': str(item.clean),
                'noisy_path': str(item.noisy),
                'enhanced_path': str(enh_path),
                'snr_in_db': snr_in,
                'snr_out_db': snr_out,
                'snr_delta_db': delta,
            })
        except Exception:
            failed += 1

    summary = {
        'run': run_name,
        'enhanced_dir': str(enh_dir),
        'num_pairs': len(items),
        'num_found_enhanced': len(rows),
        'num_missing_enhanced': missing,
        'num_failed': failed,
        **{f'snr_in_{k}': v for k, v in summarize(snr_in_values).items()},
        **{f'snr_out_{k}': v for k, v in summarize(snr_out_values).items()},
        **{f'snr_delta_{k}': v for k, v in summarize(delta_values).items()},
    }
    return summary, rows

In [ ]:
# Ausfuehrung
baseline_dir = RESULTS_ROOT / BASELINE_RUN
if not RESULTS_ROOT.is_dir():
    raise FileNotFoundError(f'results-root nicht gefunden: {RESULTS_ROOT}')
if not baseline_dir.is_dir():
    raise FileNotFoundError(f'Baseline-Run nicht gefunden: {baseline_dir}')

hp_file = baseline_dir / 'hyperparams.yaml'
hp_data_root, hp_test_ann = parse_paths_from_hyperparams(hp_file)

def _resolve_existing_file(path_like, anchors):
    p = Path(path_like).expanduser()
    candidates = []
    if p.is_absolute():
        candidates.append(p)
    else:
        candidates.append(p)
        for a in anchors:
            candidates.append((a / p).resolve())
    for c in candidates:
        if c.is_file():
            return c
    return None

def _autodetect_voicebank_root(anchors):
    checks = []
    for a in anchors:
        checks.append((a / 'data' / 'noisy-vctk-16k').resolve())
        checks.append((a / 'noisy-vctk-16k').resolve())
    seen = set()
    uniq = []
    for c in checks:
        s = str(c)
        if s not in seen:
            seen.add(s)
            uniq.append(c)
    for c in uniq:
        if (c / 'test.json').is_file():
            return c
    return None

anchors = [Path.cwd(), baseline_dir, RESULTS_ROOT.resolve()]
anchors.extend(list(Path.cwd().resolve().parents[:4]))
anchors.extend(list(RESULTS_ROOT.resolve().parents[:4]))

# Data root aufloesen (Override > hyperparams > autodetect wie metricgan)
data_root = str(DATA_ROOT_OVERRIDE) if DATA_ROOT_OVERRIDE else hp_data_root
resolved_data_root = None
if data_root:
    p = Path(data_root).expanduser()
    if p.is_absolute() and p.is_dir():
        resolved_data_root = p
    else:
        for a in anchors:
            c = (a / p).resolve() if not p.is_absolute() else p
            if c.is_dir():
                resolved_data_root = c
                break
if resolved_data_root is None:
    resolved_data_root = _autodetect_voicebank_root(anchors)

if ANNOTATION_OVERRIDE:
    ann_path = _resolve_existing_file(ANNOTATION_OVERRIDE, anchors)
else:
    ann_guess = resolve_template_path(hp_test_ann, str(resolved_data_root) if resolved_data_root else data_root)
    ann_path = _resolve_existing_file(ann_guess, anchors) if ann_guess is not None else None

# letzter Fallback: <data_root>/test.json
if ann_path is None and resolved_data_root is not None:
    fallback_ann = (resolved_data_root / 'test.json').resolve()
    if fallback_ann.is_file():
        ann_path = fallback_ann

if ann_path is None:
    raise FileNotFoundError(
        'Annotation nicht gefunden. Setze ANNOTATION_OVERRIDE oder DATA_ROOT_OVERRIDE. '
        f'Hyperparams test_annotation={hp_test_ann}, data_folder={hp_data_root}'
    )

items = load_annotation_pairs(ann_path, str(resolved_data_root) if resolved_data_root else data_root)
if not items:
    raise ValueError(f'Keine gueltigen noisy/clean Paare in {ann_path}')

run_dirs = [p for p in sorted(RESULTS_ROOT.glob(RUN_PATTERN)) if p.is_dir()]
if not run_dirs:
    raise FileNotFoundError(f'Keine Runs mit Muster {RUN_PATTERN} unter {RESULTS_ROOT}')

print('Annotation:', ann_path)
print('Data root :', resolved_data_root if resolved_data_root else data_root)
print('Pairs     :', len(items))
print('Runs      :', len(run_dirs))

summaries = []
all_rows = []

for run_dir in run_dirs:
    run_name = run_dir.name
    try:
        summary, rows = evaluate_run(run_name, run_dir, items)
        summaries.append(summary)
        all_rows.extend(rows)
        print(f'[OK] {run_name}: n={summary["snr_delta_count"]}, delta_mean={summary["snr_delta_mean"]:.3f} dB')
    except Exception as exc:
        print(f'[WARN] {run_name} uebersprungen: {exc}')

summary_df = pd.DataFrame(summaries)
details_df = pd.DataFrame(all_rows)

if summary_df.empty:
    raise RuntimeError('Keine Run-Ergebnisse auswertbar.')

if BASELINE_RUN in set(summary_df['run']):
    baseline_delta = float(summary_df.loc[summary_df['run'] == BASELINE_RUN, 'snr_delta_mean'].iloc[0])
    summary_df['delta_vs_baseline_mean'] = summary_df['snr_delta_mean'] - baseline_delta
else:
    summary_df['delta_vs_baseline_mean'] = float('nan')

summary_df = summary_df.sort_values('snr_delta_mean', ascending=False).reset_index(drop=True)

SUMMARY_CSV.parent.mkdir(parents=True, exist_ok=True)
summary_df.to_csv(SUMMARY_CSV, index=False)
details_df.to_csv(DETAILS_CSV, index=False)

display_cols = [
    'run', 'num_pairs', 'num_found_enhanced', 'num_missing_enhanced',
    'snr_in_mean', 'snr_out_mean', 'snr_delta_mean', 'delta_vs_baseline_mean'
]
display(summary_df[display_cols].round(3))
print('Summary CSV:', SUMMARY_CSV)
print('Details CSV:', DETAILS_CSV)

In [ ]:
# Visualisierung wie in der anderen Analyse
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

plot_df = summary_df.copy()
plot_df = plot_df.sort_values('snr_delta_mean', ascending=True)

axes[0].barh(plot_df['run'], plot_df['snr_delta_mean'], color='#1f77b4', alpha=0.9)
axes[0].axvline(0.0, color='black', linewidth=1)
axes[0].set_title('SNR-Verbesserung je Run')
axes[0].set_xlabel('Mean SNR Delta [dB]')
axes[0].set_ylabel('Run')

axes[1].barh(plot_df['run'], plot_df['delta_vs_baseline_mean'], color='#ff7f0e', alpha=0.9)
axes[1].axvline(0.0, color='black', linewidth=1)
axes[1].set_title(f'Delta vs Baseline ({BASELINE_RUN})')
axes[1].set_xlabel('Delta vs Baseline [dB]')
axes[1].set_ylabel('Run')

plt.tight_layout()
plt.show()

if not details_df.empty:
    plt.figure(figsize=(8, 5))
    plt.hist(details_df['snr_delta_db'].dropna().values, bins=50, alpha=0.85, color='#2ca02c')
    plt.axvline(0.0, color='black', linewidth=1)
    plt.title('Verteilung der SNR-Verbesserung (alle Utterances, alle Runs)')
    plt.xlabel('SNR Delta [dB]')
    plt.ylabel('Anzahl')
    plt.tight_layout()
    plt.show()